# 01 - Raw to Bronze
Reads USGS GeoJSON files from a Volume folder with an **explicit schema** (no inferSchema),
flattens each earthquake into one row, and **MERGEs** it into `bronze_earthquakes`.

**Parameters** (widgets at the top of the notebook):
* `source_path`  - folder that holds the raw files
* `file_pattern` - which files to process, e.g. `usgs_2024-0[1-2]-01.geojson` or `usgs_*.geojson`
* `load_type`    - `full` or `incremental` (stored on each row and in the log)
* `batch_id`     - label for this run; leave empty to auto-generate

**Idempotent:** Bronze key is (event_id, updated_ms). Re-running the same files inserts nothing new.
**Fault tolerant:** each file is processed on its own. A bad file is logged as Failure and the rest continue.

In [0]:
import glob
import uuid
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, MapType,
    TimestampType, LongType,
)

CATALOG, SCHEMA = "workspace", "seismic_pulse"
BRONZE = f"{CATALOG}.{SCHEMA}.bronze_earthquakes"
LOG_TABLE = f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"
DRIFT_TABLE = f"{CATALOG}.{SCHEMA}.schema_drift_log"

dbutils.widgets.removeAll()
dbutils.widgets.text("source_path", "/Volumes/workspace/default/seismic")
dbutils.widgets.text("file_pattern", "incremental_load_sample.geojson")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
dbutils.widgets.text("batch_id", "incr_demo")

## Explicit schema (schema-on-read)
`properties` is read as MAP<STRING,STRING>: every value arrives as text, so a type change in the source
cannot crash the read, and any brand-new column is captured automatically.
Coordinates arrive as [longitude, latitude, depth_km].

In [0]:
FEATURE_SCHEMA = StructType([
    StructField("type", StringType(), True),
    StructField("id", StringType(), True),
    StructField("properties", MapType(StringType(), StringType()), True),
    StructField("geometry", StructType([
        StructField("type", StringType(), True),
        StructField("coordinates", ArrayType(StringType()), True),
    ]), True),
])

FILE_SCHEMA = StructType([
    StructField("type", StringType(), True),
    StructField("features", ArrayType(FEATURE_SCHEMA), True),
])

# USGS property name -> Bronze column name (same order as the table)
KNOWN_PROPERTIES = {
    "mag": "mag", "place": "place", "time": "time_ms", "updated": "updated_ms",
    "tz": "tz", "url": "url", "detail": "detail", "felt": "felt", "cdi": "cdi",
    "mmi": "mmi", "alert": "alert", "status": "status", "tsunami": "tsunami",
    "sig": "sig", "net": "net", "code": "code", "ids": "ids", "sources": "sources",
    "types": "types", "nst": "nst", "dmin": "dmin", "rms": "rms", "gap": "gap",
    "magType": "mag_type", "type": "event_type", "title": "title",
}
# Columns that should hold numbers; used to detect type drift
NUMERIC_COLUMNS = ["mag", "time_ms", "updated_ms", "felt", "cdi", "mmi", "tsunami", "sig",
                   "nst", "dmin", "rms", "gap", "longitude", "latitude", "depth"]

## Logging helpers

In [0]:
LOG_SCHEMA = StructType([
    StructField("run_id", StringType()), StructField("layer", StringType()),
    StructField("load_type", StringType()), StructField("parameter", StringType()),
    StructField("start_time", TimestampType()), StructField("end_time", TimestampType()),
    StructField("status", StringType()), StructField("rows_inserted", LongType()),
    StructField("rows_updated", LongType()), StructField("rows_quarantined", LongType()),
    StructField("error_message", StringType()),
])

DRIFT_SCHEMA = StructType([
    StructField("detected_at", TimestampType()), StructField("batch_id", StringType()),
    StructField("source_file", StringType()), StructField("drift_type", StringType()),
    StructField("column_name", StringType()), StructField("detail", StringType()),
])


def log_run(run_id, layer, load_type, parameter, start, end, status,
            inserted=0, updated=0, quarantined=0, error=None):
    row = [(run_id, layer, load_type, parameter, start, end, status,
            int(inserted), int(updated), int(quarantined), error)]
    spark.createDataFrame(row, LOG_SCHEMA).write.mode("append").insertInto(LOG_TABLE)


def log_drift(rows):
    if rows:
        spark.createDataFrame(rows, DRIFT_SCHEMA).write.mode("append").insertInto(DRIFT_TABLE)

## Process one file

In [0]:
def process_file(path, load_type, batch_id):
    """Read one GeoJSON file, flatten it, check for drift, MERGE into Bronze.
    Returns the number of rows inserted."""
    raw = (spark.read
           .schema(FILE_SCHEMA)                 # explicit schema, never inferred
           .option("multiLine", True)
           .json(path))

    props = "f.properties"
    flat = (raw.select(F.explode("features").alias("f"))
            .select(
                F.col("f.id").alias("event_id"),
                *[F.expr(f"try_element_at({props}, '{src}')").alias(dst)
                  for src, dst in KNOWN_PROPERTIES.items()],
                F.expr("try_element_at(f.geometry.coordinates, 1)").alias("longitude"),
                F.expr("try_element_at(f.geometry.coordinates, 2)").alias("latitude"),
                F.expr("try_element_at(f.geometry.coordinates, 3)").alias("depth"),
                F.map_filter(F.col(props), lambda k, v: ~k.isin(list(KNOWN_PROPERTIES))).alias("extra_properties"),
            )
            .withColumn("source_file", F.lit(path))
            .withColumn("batch_id", F.lit(batch_id))
            .withColumn("load_type", F.lit(load_type))
            .withColumn("load_timestamp", F.current_timestamp()))

    if flat.limit(1).count() == 0:
        raise ValueError("no features found in file (empty or not valid GeoJSON)")

    drift_rows = []
    now = datetime.now()

    # Drift check 1: rows without an event id cannot be keyed, so they are dropped and reported
    null_keys = flat.filter(F.col("event_id").isNull()).count()
    if null_keys:
        drift_rows.append((now, batch_id, path, "NULL_KEY", "event_id", f"{null_keys} rows dropped"))
    flat = flat.filter(F.col("event_id").isNotNull())

    # Drift check 2: columns that are not in our known list (new source fields)
    new_cols = [r["k"] for r in flat.select(F.explode(F.map_keys("extra_properties")).alias("k"))
                .distinct().collect()]
    for c in new_cols:
        drift_rows.append((now, batch_id, path, "NEW_COLUMN", c, "stored in extra_properties"))

    # Drift check 3: numeric columns holding values that are not numbers
    bad = flat.agg(*[
        F.sum(F.when(F.col(c).isNotNull() & F.expr(f"try_cast({c} as double)").isNull(), 1)
              .otherwise(0)).alias(c) for c in NUMERIC_COLUMNS
    ]).first().asDict()
    for c, n in bad.items():
        if n:
            drift_rows.append((now, batch_id, path, "TYPE_DRIFT", c,
                               f"{n} non-numeric values (will be quarantined in Silver)"))
    log_drift(drift_rows)

    # Remove duplicate revisions inside the file, then MERGE (insert only if the key is new)
    src = flat.dropDuplicates(["event_id", "updated_ms"])
    src.createOrReplaceTempView("bronze_src")
    result = spark.sql(f"""
        MERGE INTO {BRONZE} AS t
        USING bronze_src AS s
        ON t.event_id = s.event_id AND t.updated_ms <=> s.updated_ms
        WHEN NOT MATCHED THEN INSERT *
    """).first()
    return result["num_inserted_rows"], result["num_updated_rows"]

## Run the pipeline for the given parameters

In [0]:
def run_raw_to_bronze(source_path, file_pattern, load_type, batch_id=""):
    batch_id = batch_id or f"batch_{datetime.now():%Y%m%d_%H%M%S}"
    run_id = str(uuid.uuid4())
    files = sorted(glob.glob(f"{source_path}/{file_pattern}"))
    print(f"batch_id={batch_id} | load_type={load_type} | files matched: {len(files)}")

    if not files:
        now = datetime.now()
        log_run(run_id, "Raw-to-Bronze", load_type, f"{source_path}/{file_pattern}", now, now,
                "Failure", error="no files matched the pattern")
        print("No files matched. Check source_path and file_pattern.")
        return

    ok, failed, total_inserted = 0, 0, 0
    for path in files:
        start = datetime.now()
        try:
            inserted, updated = process_file(path, load_type, batch_id)
            log_run(run_id, "Raw-to-Bronze", load_type, path, start, datetime.now(),
                    "Success", inserted, updated)
            ok += 1
            total_inserted += inserted
            print(f"OK   {path.split('/')[-1]}: {inserted} inserted")
        except Exception as e:
            log_run(run_id, "Raw-to-Bronze", load_type, path, start, datetime.now(),
                    "Failure", error=str(e)[:1000])
            failed += 1
            print(f"FAIL {path.split('/')[-1]}: {str(e)[:200]}")

    print(f"\nDone. Success: {ok} | Failed: {failed} | Rows inserted: {total_inserted}")


run_raw_to_bronze(
    dbutils.widgets.get("source_path"),
    dbutils.widgets.get("file_pattern"),
    dbutils.widgets.get("load_type"),
    dbutils.widgets.get("batch_id"),
)

batch_id=batch_20261010_124218 | load_type=full | files matched: 2
OK   usgs_2024-01-01.geojson: 0 inserted
OK   usgs_2024-02-01.geojson: 0 inserted

Done. Success: 2 | Failed: 0 | Rows inserted: 0


## Checks (look at these after every run)

In [0]:
b = spark.table(BRONZE)
print("Bronze rows:", b.count(),
      "| distinct (event_id, updated_ms):", b.select("event_id", "updated_ms").distinct().count())
display(spark.table(LOG_TABLE).orderBy(F.col("start_time").desc()).limit(10))
display(spark.table(DRIFT_TABLE).orderBy(F.col("detected_at").desc()).limit(10))
display(b.limit(5))

Bronze rows: 4446 | distinct (event_id, updated_ms): 4446


run_id,layer,load_type,parameter,start_time,end_time,status,rows_inserted,rows_updated,rows_quarantined,error_message
4ceffca6-b8f2-4548-9e69-86d5b659677d,Raw-to-Bronze,full,/Volumes/workspace/default/seismic/full_load_monthly/usgs_2024-02-01.geojson,2026-10-10T12:42:33.695Z,2026-10-10T12:42:39.877Z,Success,0,0,0,null
4ceffca6-b8f2-4548-9e69-86d5b659677d,Raw-to-Bronze,full,/Volumes/workspace/default/seismic/full_load_monthly/usgs_2024-01-01.geojson,2026-10-10T12:42:25.558Z,2026-10-10T12:42:31.989Z,Success,0,0,0,null
7e6fd5bb-9a79-4a67-8aa9-e87d1e345727,Bronze-to-Silver,incremental,drift_test.geojson | dates -..-,2026-10-10T12:39:46.359Z,2026-10-10T12:39:55.522Z,Success,0,0,0,null
c8029332-d0db-4469-afb1-a29922d3898f,Bronze-to-Silver,incremental,incremental_load_sample.geojson | dates -..-,2026-10-10T12:39:34.586Z,2026-10-10T12:39:44.066Z,Success,0,0,0,null
fe7ab559-5d4f-407c-a324-12b17ffa291a,Bronze-to-Silver,incremental,incremental_load_sample.geojson | dates -..-,2026-10-10T12:39:16.575Z,2026-10-10T12:39:26.975Z,Success,0,0,0,null
fe7ab559-5d4f-407c-a324-12b17ffa291a,Bronze-to-Silver,full,usgs_2024-02-01.geojson | dates -..-,2026-10-10T12:39:03.062Z,2026-10-10T12:39:15.142Z,Success,0,0,0,null
fe7ab559-5d4f-407c-a324-12b17ffa291a,Bronze-to-Silver,full,usgs_2024-01-01.geojson | dates -..-,2026-10-10T12:38:51.458Z,2026-10-10T12:39:01.700Z,Success,0,0,0,null
fe7ab559-5d4f-407c-a324-12b17ffa291a,Bronze-to-Silver,incremental,drift_test.geojson | dates -..-,2026-10-10T12:38:40.434Z,2026-10-10T12:38:50.047Z,Success,0,0,0,null
831777bc-7e01-4ce5-999e-f6bb4dd9fd8b,Bronze-to-Silver,incremental,drift_test.geojson | dates -..-,2026-10-10T12:38:13.634Z,2026-10-10T12:38:23.646Z,Success,0,0,0,null
3571ca74-5857-4687-b1ba-910219e97f10,Bronze-to-Silver,incremental,drift_test.geojson | dates -..-,2026-10-10T12:19:27.407Z,2026-10-10T12:19:37.295Z,Success,0,0,0,null


detected_at,batch_id,source_file,drift_type,column_name,detail
2026-10-10T11:33:16.905Z,drift_test_1,/Volumes/workspace/default/seismic/drift_test/drift_test.geojson,NULL_KEY,event_id,1 rows dropped
2026-10-10T11:33:16.905Z,drift_test_1,/Volumes/workspace/default/seismic/drift_test/drift_test.geojson,NEW_COLUMN,brand_new_field,stored in extra_properties
2026-10-10T11:33:16.905Z,drift_test_1,/Volumes/workspace/default/seismic/drift_test/drift_test.geojson,TYPE_DRIFT,mag,1 non-numeric values (will be quarantined in Silver)


event_id,mag,place,time_ms,updated_ms,tz,url,detail,felt,cdi,mmi,alert,status,tsunami,sig,net,code,ids,sources,types,nst,dmin,rms,gap,mag_type,event_type,title,longitude,latitude,depth,extra_properties,source_file,batch_id,load_type,load_timestamp
hv74082126,2.74000001,"6 km SW of Volcano, Hawaii",1706750589150,1713036681040,null,https://earthquake.usgs.gov/earthquakes/eventpage/hv74082126,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=hv74082126&format=geojson,null,null,null,null,automatic,0,116,hv,74082126,",us7000lvkw,hv74082126,",",us,hv,",",origin,phase-data,",28,0.002392,0.230000004,70,md,earthquake,"M 2.7 - 6 km SW of Volcano, Hawaii",-155.274993896484,19.3933334350586,1.5,Map(),/Volumes/workspace/default/seismic/full_load_monthly/usgs_2024-02-01.geojson,batch_20261010_084253,full,2026-10-10T08:43:35.933Z
hv74082226,2.88,"8 km SW of Volcano, Hawaii",1706751776020,1713036686040,null,https://earthquake.usgs.gov/earthquakes/eventpage/hv74082226,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=hv74082226&format=geojson,2,2,null,null,reviewed,0,128,hv,74082226,",hv74082226,us7000lvlb,",",hv,us,",",dyfi,origin,phase-data,",28,0.001711,0.09,44,ml,earthquake,"M 2.9 - 8 km SW of Volcano, Hawaii",-155.282,19.3848333333333,1.25,Map(),/Volumes/workspace/default/seismic/full_load_monthly/usgs_2024-02-01.geojson,batch_20261010_084253,full,2026-10-10T08:43:35.933Z
nc73997246,2.62,"2 km E of Canyondam, CA",1706779948790,1713036738040,null,https://earthquake.usgs.gov/earthquakes/eventpage/nc73997246,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=nc73997246&format=geojson,1,2.2,null,null,reviewed,0,106,nc,73997246,",nc73997246,us7000lvrf,",",nc,us,",",dyfi,nearby-cities,origin,phase-data,scitech-link,",21,0.1074,0.07,73,md,earthquake,"M 2.6 - 2 km E of Canyondam, CA",-121.0503333,40.1686667,4.41,Map(),/Volumes/workspace/default/seismic/full_load_monthly/usgs_2024-02-01.geojson,batch_20261010_084253,full,2026-10-10T08:43:35.933Z
us7000lvrk,4,"24 km N of Murghob, Tajikistan",1706780623986,1713036739040,null,https://earthquake.usgs.gov/earthquakes/eventpage/us7000lvrk,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us7000lvrk&format=geojson,null,null,null,null,reviewed,0,246,us,7000lvrk,",us7000lvrk,",",us,",",origin,phase-data,",41,1.982,0.73,86,mb,earthquake,"M 4.0 - 24 km N of Murghob, Tajikistan",73.9207,38.3846,133.724,Map(),/Volumes/workspace/default/seismic/full_load_monthly/usgs_2024-02-01.geojson,batch_20261010_084253,full,2026-10-10T08:43:35.933Z
hv74084086,2.64,"11 km SSW of Volcano, Hawaii",1706789730570,1713036749040,null,https://earthquake.usgs.gov/earthquakes/eventpage/hv74084086,https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=hv74084086&format=geojson,null,null,null,null,automatic,0,107,hv,74084086,",us7000lvtg,hv74084086,",",us,hv,",",origin,phase-data,",20,0.03173,0.109999999,64,ml,earthquake,"M 2.6 - 11 km SSW of Volcano, Hawaii",-155.293502807617,19.3566665649414,0.939999997615814,Map(),/Volumes/workspace/default/seismic/full_load_monthly/usgs_2024-02-01.geojson,batch_20261010_084253,full,2026-10-10T08:43:35.933Z


In [0]:
run_raw_to_bronze(
    "/Volumes/workspace/default/seismic",     # source_path
    "incremental_load_sample.geojson",        # file_pattern
    "incremental",                            # load_type
    "incr_demo",                              # batch_id
)

batch_id=incr_demo | load_type=incremental | files matched: 1
OK   incremental_load_sample.geojson: 0 inserted

Done. Success: 1 | Failed: 0 | Rows inserted: 0


In [0]:
run_raw_to_bronze("/Volumes/workspace/default/seismic/drift_test", "drift_test.geojson", "incremental", "drift_test_1")

batch_id=drift_test_1 | load_type=incremental | files matched: 1
OK   drift_test.geojson: 5 inserted

Done. Success: 1 | Failed: 0 | Rows inserted: 5


In [0]:
BASE = "/Volumes/workspace/default/seismic/full_load_monthly"
for y in range(2015, 2018):          # start with 2015-2017 to gauge the time
    run_raw_to_bronze(BASE, f"usgs_{y}-*.geojson", "full", f"full_load_{y}")

batch_id=full_load_2015 | load_type=full | files matched: 12
OK   usgs_2015-01-01.geojson: 2220 inserted
OK   usgs_2015-02-01.geojson: 1946 inserted
OK   usgs_2015-03-01.geojson: 2116 inserted
OK   usgs_2015-04-01.geojson: 2012 inserted
OK   usgs_2015-05-01.geojson: 2439 inserted
OK   usgs_2015-06-01.geojson: 1948 inserted
OK   usgs_2015-07-01.geojson: 2573 inserted
OK   usgs_2015-08-01.geojson: 2285 inserted
OK   usgs_2015-09-01.geojson: 2975 inserted
OK   usgs_2015-10-01.geojson: 2138 inserted
OK   usgs_2015-11-01.geojson: 2341 inserted
OK   usgs_2015-12-01.geojson: 2061 inserted

Done. Success: 12 | Failed: 0 | Rows inserted: 27054
batch_id=full_load_2016 | load_type=full | files matched: 12
OK   usgs_2016-01-01.geojson: 2290 inserted
OK   usgs_2016-02-01.geojson: 2008 inserted
OK   usgs_2016-03-01.geojson: 1894 inserted
OK   usgs_2016-04-01.geojson: 2378 inserted
OK   usgs_2016-05-01.geojson: 1979 inserted
OK   usgs_2016-06-01.geojson: 1984 inserted
OK   usgs_2016-07-01.geojson: 19